In [1]:
import os, pickle
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
class DenseRFF_PT(nn.Module):
    def __init__(self, Nf, scale=None, gamma=None, normalization=True, function="cos", trainable_scale=True, trainable_W=True, seed=None, kernel='gaussian'):
        super().__init__()
        self.Nf = Nf
        self.gamma = gamma
        self.scale = scale
        self.normalization = normalization
        self.function = function
        self.trainable_scale = trainable_scale
        self.trainable_W = trainable_W
        self.seed = seed
        self.kernel_type = kernel
        self.W = None
        self.b = None
        self.rho_scale = None
        self._eps = 1e-8
        self.bandwidth_history = []

    def _get_random_features_initializer(self, shape, sigma=1.0, seed=None):
        if seed is not None:
            np.random.seed(seed)
        if self.kernel_type == 'gaussian':
            return np.random.randn(*shape) / sigma
        elif self.kernel_type == 'laplacian':
            return np.random.laplace(loc=0.0, scale=1.0, size=shape) / sigma
        else:
            raise ValueError(f'Unsupported initializer {self.kernel_type}')

    def _ensure_params_initialized(self, device, D):
        if self.W is None:
            if self.gamma is not None:
                sigma = np.sqrt(1.0 / (2 * self.gamma))
            else:
                sigma = 1.0
            if self.scale is None:
                self.scale = sigma
            W_init = self._get_random_features_initializer((D, self.Nf), sigma=self.scale, seed=self.seed)
            self.W = nn.Parameter(torch.tensor(W_init, dtype=torch.float32, device=device), requires_grad=self.trainable_W)
            b_init = np.random.uniform(0.0, 2 * np.pi, size=(self.Nf,))
            self.b = nn.Parameter(torch.tensor(b_init, dtype=torch.float32, device=device), requires_grad=self.trainable_W)
            init_kernel_scale = 1.0
            rho0 = np.log(np.exp(init_kernel_scale) - 1.0)
            self.rho_scale = nn.Parameter(torch.tensor([rho0], dtype=torch.float32, device=device), requires_grad=self.trainable_scale)

    def _kernel_scale(self):
        return F.softplus(self.rho_scale) + self._eps

    def bandwidth_lengthscale(self):
        if self.rho_scale is None or self.scale is None:
            return None
        ks = float(self._kernel_scale().detach().cpu().item())
        return float(self.scale) / ks

    @torch.no_grad()
    def log_bandwidth(self, step):
        ell = self.bandwidth_lengthscale()
        if ell is not None:
            self.bandwidth_history.append((step, float(ell)))

    def forward(self, inputs):
        device = inputs.device
        if inputs.dim() == 2:
            inputs = inputs.unsqueeze(1)
        elif inputs.dim() != 3:
            raise ValueError(f"Expected [B,T,D], got {inputs.shape}")
        B, T, D = inputs.shape
        self._ensure_params_initialized(device, D)
        kernel_scale = self._kernel_scale()
        proj = torch.matmul(inputs, self.W * kernel_scale) + self.b
        outputs = torch.cos(proj) * np.sqrt(2.0 / self.Nf)
        if self.normalization:
            norm = np.sqrt(self.Nf)
            outputs = outputs / norm
        return outputs.permute(0, 2, 1)

class SpectralDropout1d(nn.Module):
    def __init__(self, p: float = 0.1, channels_last: bool = True):
        super().__init__()
        self.p = p
        self.channels_last = channels_last
    def forward(self, x):
        if (not self.training) or self.p == 0.0:
            return x
        if self.channels_last:
            B, T, F = x.shape
            mask = (torch.rand(B, 1, F, device=x.device) > self.p).float() / (1.0 - self.p)
            return x * mask
        else:
            B, F, T = x.shape
            mask = (torch.rand(B, F, 1, device=x.device) > self.p).float() / (1.0 - self.p)
            return x * mask

class TemporalSpectralBlock(nn.Module):
    def __init__(self, features, kernel_size=3, dilation=2, p_drop=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(features, features, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(features, features, kernel_size, padding=padding, dilation=1)
        self.norm = nn.LayerNorm(features)
        self.act = nn.GELU()
        self.drop = nn.Dropout(p_drop)
    def forward(self, x):
        res = x
        y = x.transpose(1, 2)
        y = self.act(self.conv1(y))
        y = self.conv2(y).transpose(1, 2)
        if y.size(1) != res.size(1):
            min_len = min(y.size(1), res.size(1))
            y = y[:, :min_len, :]
            res = res[:, :min_len, :]
        y = self.norm(y + res)
        y = self.act(y)
        return self.drop(y)

class MultiBandRFFEncoder(nn.Module):
    def __init__(self, in_dim=1, bands=(4., 24., 168.), nf_per_band=32, kernel="gaussian", spectral_dropout_p=0.1):
        super().__init__()
        self.bands = nn.ModuleList([
            DenseRFF_PT(Nf=nf_per_band, function="cos", trainable_W=True, trainable_scale=True, kernel=kernel, scale=band)
            for band in bands
        ])
        self.out_dim = nf_per_band * len(self.bands)
        self.norm = nn.LayerNorm(self.out_dim)
        self.spec_do = SpectralDropout1d(p=spectral_dropout_p, channels_last=True)
        self.drop = nn.Dropout(spectral_dropout_p)
    def forward(self, x):
        outs = []
        for b in self.bands:
            z = b(x)
            z = z.transpose(1, 2)
            outs.append(z)
        zcat = torch.cat(outs, dim=2)
        zcat = self.norm(zcat)
        zcat = self.drop(self.spec_do(zcat))
        return zcat

class TemporalHead(nn.Module):
    def __init__(self, hidden, horizon, pool="last"):
        super().__init__()
        self.pool = pool
        self.fc1 = nn.Linear(hidden, hidden // 2)
        self.fc2 = nn.Linear(hidden // 2, horizon)
        self.act = nn.ReLU()
    def forward(self, H):
        if self.pool == "mean":
            v = H.mean(dim=1)
        else:
            v = H[:, -1, :]
        return self.fc2(self.act(self.fc1(v)))

class RFF_AnyRNN_Forecaster(nn.Module):
    def __init__(self, horizon=24, rnn_type="RNN", bands=(4., 24., 168.), nf_per_band=64, hidden=96, num_layers=1, bidirectional=False, use_tsb=True, kernel="gaussian", spectral_dropout_p=0.1, pool="last"):
        super().__init__()
        self.enc = MultiBandRFFEncoder(1, bands, nf_per_band, kernel, spectral_dropout_p)
        self.tsb = TemporalSpectralBlock(self.enc.out_dim) if use_tsb else nn.Identity()
        self.rnn = getattr(nn, rnn_type)(input_size=self.enc.out_dim, hidden_size=hidden, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        out_hidden = hidden * (2 if bidirectional else 1)
        self.head = TemporalHead(out_hidden, horizon, pool)
    def forward(self, x):
        z = self.enc(x)
        z = self.tsb(z)
        H, _ = self.rnn(z)
        return self.head(H)

In [3]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
which = "Synthetic" # Synthetic, Argonne
path = os.path.join(ROOT, "results", "models", f"best_model_{which}.pth")

cfg = {
    "Argonne": dict(horizon=7, rnn_type="LSTM", bands=(12., 24., 168.), nf_per_band=104, hidden=144, num_layers=3, bidirectional=False, use_tsb=True, kernel="gaussian", spectral_dropout_p=0.2, pool="last"),
    "Synthetic": dict(horizon=24, rnn_type="LSTM", bands=(4., 24., 168.), nf_per_band=16, hidden=176, num_layers=1, bidirectional=False, use_tsb=True, kernel="gaussian", spectral_dropout_p=0.3, pool="last"),
}
model = RFF_AnyRNN_Forecaster(**cfg[which]).to(device)
dummy = torch.zeros(1, 168, 1, device=device)
_ = model(dummy)
model.load_state_dict(torch.load(path, map_location=device))
model.eval()

RFF_AnyRNN_Forecaster(
  (enc): MultiBandRFFEncoder(
    (bands): ModuleList(
      (0-2): 3 x DenseRFF_PT()
    )
    (norm): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (spec_do): SpectralDropout1d()
    (drop): Dropout(p=0.3, inplace=False)
  )
  (tsb): TemporalSpectralBlock(
    (conv1): Conv1d(48, 48, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
    (conv2): Conv1d(48, 48, kernel_size=(3,), stride=(1,), padding=(2,))
    (norm): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (act): GELU(approximate='none')
    (drop): Dropout(p=0.1, inplace=False)
  )
  (rnn): LSTM(48, 176, batch_first=True)
  (head): TemporalHead(
    (fc1): Linear(in_features=176, out_features=88, bias=True)
    (fc2): Linear(in_features=88, out_features=24, bias=True)
    (act): ReLU()
  )
)

In [4]:
pkl_path = os.path.join(ROOT, "data", "processed", "data_dict.pkl")
with open(pkl_path, "rb") as f:
    data_dict_loaded = pickle.load(f)

data_key = which if which in data_dict_loaded else ("Argone" if which == "Argonne" else which)
X = data_dict_loaded[data_key]["X"]
Y = data_dict_loaded[data_key]["Y"]
X = X[..., np.newaxis]

X_temp, X_test, y_temp, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

batch_size = 256
def make_loader(X_arr, y_arr, batch_size, shuffle=False):
    ds = TensorDataset(torch.tensor(X_arr, dtype=torch.float32), torch.tensor(y_arr, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, batch_size)
valid_loader = make_loader(X_valid, y_valid, batch_size)
test_loader = make_loader(X_test, y_test, batch_size)

In [5]:
# Índice de la ventana de test (cambiar entre 0 y len(X_test)-1)
window_index = 3

x_one = torch.tensor(X_test[window_index : window_index + 1], dtype=torch.float32, device=device)
y_true_one = y_test[window_index]
with torch.no_grad():
    y_pred_one = model(x_one)
y_pred_one = y_pred_one.cpu().numpy().squeeze()

In [6]:
with torch.no_grad():
    z_rff = model.enc(x_one)
z_rff = z_rff.cpu().numpy().squeeze()

In [7]:
# Directorio de salida y datos para las figuras (reconstrucción, importancia, RFF)
rff_plot_dir = os.path.join(ROOT, "results", "plots", "RFFs")
os.makedirs(rff_plot_dir, exist_ok=True)
x_plot = x_one.cpu().numpy().squeeze()
nf_per_band = z_rff.shape[1] // 3
z_band0 = z_rff[:, :nf_per_band]
z_band1 = z_rff[:, nf_per_band : 2 * nf_per_band]
z_band2 = z_rff[:, 2 * nf_per_band :]
vmin = min(z_band0.min(), z_band1.min(), z_band2.min())
vmax = max(z_band0.max(), z_band1.max(), z_band2.max())


In [8]:
import time

model.eval()
steps = 20
# Sensibilidad: importancia IG sobre el total del conjunto de test (todas las ventanas)
sens_indices = np.arange(len(X_test))
imp_pct_list = []

progress_every = 10  # imprime estado cada N ventanas
n_total = int(len(sens_indices))
t0 = time.perf_counter()

for k, idx in enumerate(sens_indices, start=1):
    if (k == 1) or (k % progress_every == 0) or (k == n_total):
        elapsed = time.perf_counter() - t0
        rate = elapsed / max(k, 1)
        eta = rate * (n_total - k)
        print(f"[IG] ventana {k}/{n_total} | elapsed={elapsed/60:.1f} min | ETA={eta/60:.1f} min")

    x_w = torch.tensor(X_test[idx : idx + 1], dtype=torch.float32, device=device)
    z_input = model.enc(x_w).detach()
    baseline = torch.zeros_like(z_input)
    acc_grad = torch.zeros_like(z_input)
    for alpha in torch.linspace(0.0, 1.0, steps, device=z_input.device):
        z_alpha = baseline + alpha * (z_input - baseline)
        z_alpha.requires_grad_(True)
        h = model.tsb(z_alpha)
        H, _ = model.rnn(h)
        y = model.head(H)
        target = y.mean()
        grad = torch.autograd.grad(target, z_alpha, retain_graph=False, create_graph=False)[0]
        acc_grad += grad
    avg_grad = acc_grad / steps
    ig = (z_input - baseline) * avg_grad
    nf = ig.shape[-1] // 3
    ig_abs = ig.abs()[0]
    imp_1 = ig_abs[:, :nf].sum().item()
    imp_2 = ig_abs[:, nf:2*nf].sum().item()
    imp_3 = ig_abs[:, 2*nf:].sum().item()
    imp = np.array([imp_1, imp_2, imp_3], dtype=np.float64)
    imp_pct_i = 100.0 * imp / (imp.sum() + 1e-12)
    imp_pct_list.append(imp_pct_i)
imp_pct_all = np.array(imp_pct_list)
imp_pct = imp_pct_all.mean(axis=0)
imp_pct_std = imp_pct_all.std(axis=0)

total_elapsed = time.perf_counter() - t0
print(f"[IG] listo | total={total_elapsed/60:.1f} min | ventanas={n_total} | steps={steps}")


[IG] ventana 1/1128 | elapsed=0.0 min | ETA=0.0 min
[IG] ventana 10/1128 | elapsed=0.0 min | ETA=1.8 min
[IG] ventana 20/1128 | elapsed=0.0 min | ETA=1.7 min
[IG] ventana 30/1128 | elapsed=0.0 min | ETA=1.6 min
[IG] ventana 40/1128 | elapsed=0.1 min | ETA=1.6 min
[IG] ventana 50/1128 | elapsed=0.1 min | ETA=1.6 min
[IG] ventana 60/1128 | elapsed=0.1 min | ETA=1.5 min
[IG] ventana 70/1128 | elapsed=0.1 min | ETA=1.5 min
[IG] ventana 80/1128 | elapsed=0.1 min | ETA=1.4 min
[IG] ventana 90/1128 | elapsed=0.1 min | ETA=1.4 min
[IG] ventana 100/1128 | elapsed=0.1 min | ETA=1.4 min
[IG] ventana 110/1128 | elapsed=0.1 min | ETA=1.3 min
[IG] ventana 120/1128 | elapsed=0.2 min | ETA=1.3 min
[IG] ventana 130/1128 | elapsed=0.2 min | ETA=1.3 min
[IG] ventana 140/1128 | elapsed=0.2 min | ETA=1.3 min
[IG] ventana 150/1128 | elapsed=0.2 min | ETA=1.3 min
[IG] ventana 160/1128 | elapsed=0.2 min | ETA=1.2 min
[IG] ventana 170/1128 | elapsed=0.2 min | ETA=1.2 min
[IG] ventana 180/1128 | elapsed=0.2 min

In [9]:
# Mapas de RFF por banda (1x3, una sola figura)
z_bands = [z_band0, z_band1, z_band2]

# En este notebook las 3 escalas vienen desde `cfg[which]["bands"]`.
# Las reportamos como rho (LaTeX) para etiquetar.
try:
    rho_vals = cfg[which].get("bands", None)
except Exception:
    rho_vals = None

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

m_last = None

import matplotlib.ticker as mtick

for band_idx, (ax, z_b) in enumerate(zip(axes, z_bands)):
    n_hours, nf = z_b.shape[0], z_b.shape[1]
    x_edges = np.arange(n_hours + 1)
    y_edges = np.arange(nf + 1)

    m = ax.pcolormesh(
        x_edges,
        y_edges,
        z_b.T,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        shading="auto",
    )
    m_last = m

    ax.set_aspect("auto")
    ax.set_xlabel("Hour", size=16)
    if band_idx == 0:
        ax.set_ylabel("Feature", size=16)
    else:
        ax.set_yticklabels([])

    ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))

    rho = None
    if rho_vals is not None and len(rho_vals) > band_idx:
        rho = rho_vals[band_idx]

    if rho is not None:
        ax.set_title(rf"RFF ($\rho_{{{band_idx+1}}}={rho:.3g}$)", size=14)
    else:
        ax.set_title(rf"RFF ($\rho_{{{band_idx+1}}}$)", size=14)

# Un solo colorbar en la columna derecha
fig.colorbar(m_last, ax=axes[2], pad=0.02, shrink=0.95, aspect=15, location="right")

plt.tight_layout()

out_path = os.path.join(rff_plot_dir, f"{which}_rff_bands_1x3.eps")
print(f"[layers_figures] guardando: {out_path}")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()


ValueError: 
RFF ($\\rho_1$=4)
     ^
ParseException: Expected end of text, found '$'  (at char 5), (line:1, col:6)

Error in callback <function _draw_all_if_interactive at 0x11b847c40> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
RFF ($\\rho_1$=4)
     ^
ParseException: Expected end of text, found '$'  (at char 5), (line:1, col:6)

ValueError: 
RFF ($\\rho_1$=4)
     ^
ParseException: Expected end of text, found '$'  (at char 5), (line:1, col:6)

<Figure size 1200x350 with 4 Axes>